# Netflix Data Analysis — Week 2 (DawoodTech)

End-to-end EDA notebook for the Netflix titles dataset.

**Sections:**
1. Data Loading & Overview
2. Data Cleaning
3. Data Analysis
4. Correlation Analysis
5. Key Insights

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)

VIZ_DIR = os.path.join('..', 'visualizations')
os.makedirs(VIZ_DIR, exist_ok=True)
DATA_PATH = os.path.join('..', 'data', 'netflix_titles.csv')
print('Reading:', DATA_PATH)

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
df.dtypes

## 2. Data Cleaning

Steps applied:
- Drop exact duplicate rows.
- Fill string columns (`director`, `cast`, `country`, `rating`) with `Unknown` — losing the row would lose ~10% of the data and the remaining columns are still useful.
- Drop rows with missing `date_added` (a few percent only — those rows can't contribute to any time-series view anyway).
- Rename columns to snake_case (already snake_case in the source; we normalize defensively).
- Parse `date_added` into a real `datetime`, then derive `year_added` and `month_added`.
- Split `duration` into a numeric `duration_value` and a textual `duration_unit`.

In [ ]:
before_rows = len(df)
missing_before = df.isna().sum().rename('missing_before')

df = df.drop_duplicates().copy()
print(f'Removed {before_rows - len(df)} duplicate rows')

df.columns = (
    df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')
)
print('Columns:', list(df.columns))

In [ ]:
for col in ['director', 'cast', 'country', 'rating']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown').replace('', 'Unknown')

df['date_added'] = pd.to_datetime(df['date_added'].astype(str).str.strip(), errors='coerce')
df = df.dropna(subset=['date_added']).copy()

df['year_added'] = df['date_added'].dt.year.astype(int)
df['month_added'] = df['date_added'].dt.month.astype(int)

df['duration'] = df['duration'].fillna('').astype(str)
split = df['duration'].str.extract(r'(?P<duration_value>\d+)\s*(?P<duration_unit>[A-Za-z]+)?')
df['duration_value'] = pd.to_numeric(split['duration_value'], errors='coerce')
df['duration_unit'] = split['duration_unit'].fillna('').str.lower().str.rstrip('s')

missing_after = df.isna().sum().rename('missing_after')
pd.concat([missing_before, missing_after], axis=1).fillna(0).astype(int)

In [ ]:
print('Final shape:', df.shape)
df.head()

## 3. Data Analysis

In [ ]:
# Movies vs TV Shows
type_counts = df['type'].value_counts()
type_counts

In [ ]:
# Top 10 countries — split multi-country rows on comma
country_series = df['country'].str.split(', ').explode().str.strip()
country_series = country_series[country_series != 'Unknown']
top_countries = country_series.value_counts().head(10)
top_countries

In [ ]:
# Most common ratings
rating_counts = df['rating'].value_counts().head(10)
rating_counts

In [ ]:
# Yearly content addition trend
yearly = df.groupby('year_added').size().sort_index()
yearly

In [ ]:
# Top 10 genres
genre_series = df['listed_in'].str.split(', ').explode().str.strip()
top_genres = genre_series.value_counts().head(10)
top_genres

In [ ]:
# Average movie duration by rating
movies = df[(df['type'] == 'Movie') & (df['duration_unit'] == 'min')].copy()
avg_duration_by_rating = (
    movies.groupby('rating')['duration_value'].mean().sort_values(ascending=False)
)
avg_duration_by_rating

In [ ]:
# Month-wise content addition pattern
monthly = df.groupby('month_added').size()
monthly.index = [pd.Timestamp(2000, m, 1).strftime('%b') for m in monthly.index]
monthly

## 4. Correlation Analysis

In [ ]:
num_df = df[['release_year', 'year_added', 'month_added', 'duration_value']].copy()
corr = num_df.corr()
corr

In [ ]:
# --- Render and save all six visualizations ---

# 1. Top 10 countries bar chart
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, ax=ax, palette='viridis')
ax.set_title('Top 10 Countries by Netflix Content')
ax.set_xlabel('Number of titles')
ax.set_ylabel('Country')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '01_top_countries.png'), dpi=120)
plt.show()

In [ ]:
# 2. Movies vs TV Shows pie chart
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    colors=['#E50914', '#221f1f'],
    startangle=90,
    textprops={'color': 'white', 'fontsize': 14},
)
ax.set_title('Movies vs TV Shows')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '02_movies_vs_tv.png'), dpi=120)
plt.show()

In [ ]:
# 3. Yearly content trend line chart (2010-2021)
yearly_window = yearly.loc[(yearly.index >= 2010) & (yearly.index <= 2021)]
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(yearly_window.index, yearly_window.values, marker='o', linewidth=2.5, color='#E50914')
ax.fill_between(yearly_window.index, yearly_window.values, alpha=0.15, color='#E50914')
ax.set_title('Netflix Content Added per Year (2010–2021)')
ax.set_xlabel('Year added')
ax.set_ylabel('Titles added')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '03_yearly_trend.png'), dpi=120)
plt.show()

In [ ]:
# 4. Movie duration histogram
fig, ax = plt.subplots(figsize=(11, 6))
sns.histplot(movies['duration_value'].dropna(), bins=30, kde=True, color='#564d4d', ax=ax)
ax.set_title('Movie Duration Distribution (minutes)')
ax.set_xlabel('Duration (minutes)')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '04_movie_duration_hist.png'), dpi=120)
plt.show()

In [ ]:
# 5. Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation — Numeric Columns')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '05_correlation_heatmap.png'), dpi=120)
plt.show()

In [ ]:
# 6. Top 10 genres horizontal bar chart
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, ax=ax, palette='mako')
ax.set_title('Top 10 Genres on Netflix')
ax.set_xlabel('Number of titles')
ax.set_ylabel('Genre')
fig.tight_layout()
fig.savefig(os.path.join(VIZ_DIR, '06_top_genres.png'), dpi=120)
plt.show()

## 5. Key Insights

- **Movies dominate the catalog.** Roughly 70% of all titles are movies, with TV shows the remaining ~30% — Netflix's library is still movie-heavy despite growing series investment.
- **The United States and India lead production.** The US contributes the largest share of titles, followed by India and the UK; the top 3 countries account for a disproportionate slice of the catalog.
- **Catalog growth peaked around 2018–2020.** Yearly additions climb sharply from 2015, peak in 2019–2020, then taper — consistent with Netflix's aggressive originals expansion ending around 2020.
- **Mature ratings dominate.** TV-MA and TV-14 are the two most common ratings, signaling that adult-oriented content drives library size more than family content.
- **Movies cluster around 90–110 minutes.** The duration histogram peaks in the standard feature-film range; very few movies exceed 150 minutes.
- **Weak numeric correlations.** `release_year` and `year_added` correlate moderately (newer content gets added sooner), but other numeric fields show little linear relationship — most signal in this dataset is categorical.